# Circuits — the components, one at a time

Before any circuit does anything useful, five parts have to be understood on their own. Each one is a rule tying the voltage across it to the current through it, and every rule below is a different *kind* of relationship:

$$v=Ri \qquad i=C\frac{dv}{dt} \qquad v=L\frac{di}{dt} \qquad i=I_s\!\left(e^{v/nV_T}-1\right)$$

A resistor's rule is **algebraic** — the current now depends on the voltage now. A capacitor's and an inductor's are **differential**, so they depend on history and that is what gives a circuit memory. A diode's is **nonlinear**, which is what lets circuits do something other than scale and delay.

The schematics animate in the style of Falstad's simulator: wires are coloured by their node voltage, and the yellow dots are charge in motion — their speed is the current, and their direction is its sign. When nothing moves, no current flows.

Press ▶ in any section, or drag the time slider. Everything is integrated numerically and checked against the closed-form answer where one exists.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import ipywidgets as widgets
from IPython.display import display

BG, PANEL, FG = "#05070b", "#0a0d14", "#c9cfda"
MUTED, GRIDC = "#6b7280", "#1b2130"
POS, NEG, DOT = "#3fd0c9", "#e0555c", "#ffd24a"
BLUE, ORANGE, GREEN, PURP = "#5aa9e6", "#e08a3c", "#7ddc7d", "#b48ce0"
VMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "volt", [(0.0, NEG), (0.5, "#4a5060"), (1.0, POS)])

plt.rcParams.update({
    "figure.dpi": 112, "font.size": 8.5, "axes.titlesize": 9,
    "figure.facecolor": BG, "savefig.facecolor": BG, "axes.facecolor": PANEL,
    "axes.edgecolor": GRIDC, "axes.labelcolor": FG, "text.color": FG,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRIDC,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.facecolor": PANEL, "legend.edgecolor": GRIDC, "legend.framealpha": 0.9,
})
SL = {"style": {"description_width": "104px"},
      "layout": widgets.Layout(width="290px"), "continuous_update": False}


def panel(ax, edge=None, lw=1.3):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(edge or GRIDC)
        s.set_linewidth(lw if edge else 0.8)
    ax.tick_params(colors=MUTED, labelsize=7)
    return ax


def readout(fig, x, y, lines, color=FG, size=7.4):
    fig.text(x, y, "\n".join(lines), family="monospace", fontsize=size,
             color=color, va="top", ha="left", linespacing=1.55)


def footer(fig, text):
    fig.text(0.010, 0.012, text, family="monospace", fontsize=6.6, color=MUTED)
    fig.text(0.990, 0.012, "circuits · components", family="monospace",
             fontsize=6.6, color=MUTED, ha="right")


def timeline(n, step=1, interval=90, desc="time"):
    p = widgets.Play(value=0, min=0, max=n, step=step, interval=interval)
    s = widgets.IntSlider(value=0, min=0, max=n, step=step, description=desc + ":",
                          continuous_update=False,
                          style={"description_width": "104px"},
                          layout=widgets.Layout(width="430px"))
    widgets.jslink((p, "value"), (s, "value"))
    return p, s


# ----- schematic primitives, Falstad style -------------------------------
def vcolor(v, vmax):
    return VMAP(np.clip(0.5 + 0.5 * v / max(vmax, 1e-9), 0, 1))


def wire(ax, pts, v, vmax, lw=2.6):
    pts = np.asarray(pts, float)
    seg = np.stack([pts[:-1], pts[1:]], axis=1)
    ax.add_collection(LineCollection(seg, colors=[vcolor(v, vmax)] * len(seg),
                                     linewidths=lw, zorder=2))


def node_dot(ax, p, v, vmax, s=34):
    ax.plot(*p, "o", ms=np.sqrt(s), color=vcolor(v, vmax), zorder=4)


def resistor(ax, p0, p1, v, vmax, label=None, n=6, amp=0.16):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.28, p1 - u * L * 0.28
    ts = np.linspace(0, 1, 2 * n + 1)
    zz = [a + (b - a) * t + nrm * amp * ((-1) ** k if 0 < k < 2 * n else 0)
          for k, t in enumerate(ts)]
    wire(ax, [p0, a], v, vmax)
    wire(ax, zz, v, vmax, lw=2.2)
    wire(ax, [b, p1], v, vmax)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.34), label, color=FG, fontsize=7.5,
                ha="center", va="center")


def capacitor(ax, p0, p1, v, vmax, label=None, gap=0.10, half=0.24):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * gap, c + u * gap
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    for q in (a, b):
        ax.plot(*np.stack([q - nrm * half, q + nrm * half]).T, color=FG, lw=2.4,
                zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def inductor(ax, p0, p1, v, vmax, label=None, coils=4, r=0.13):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.25, p1 - u * L * 0.25
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    seg = np.linalg.norm(b - a) / coils
    for k in range(coils):
        c = a + u * seg * (k + 0.5)
        th = np.linspace(0, np.pi, 24)
        pts = np.array([c + u * (seg / 2) * np.cos(np.pi - t) + nrm * r * np.sin(t)
                        for t in th])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=2.0, zorder=3)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.38), label, color=FG, fontsize=7.5,
                ha="center")


def diode(ax, p0, p1, v, vmax, label=None, s=0.20):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * s, c + u * s
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    ax.add_patch(mpatches.Polygon([a + nrm * s, a - nrm * s, b], closed=True,
                                  facecolor=ORANGE, edgecolor=ORANGE, zorder=3))
    ax.plot(*np.stack([b - nrm * s, b + nrm * s]).T, color=FG, lw=2.6, zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def source(ax, p0, p1, v, vmax, kind="dc", label=None, r=0.30):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    c = 0.5 * (p0 + p1)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    wire(ax, [p0, c - u * r], v, vmax); wire(ax, [c + u * r, p1], v, vmax)
    ax.add_patch(mpatches.Circle(c, r, fill=False, ec=FG, lw=2.0, zorder=3))
    if kind == "dc":
        ax.plot(*np.stack([c - u * 0.12 - nrm * 0.16, c - u * 0.12 + nrm * 0.16]).T,
                color=FG, lw=2.6, zorder=4)
        ax.plot(*np.stack([c + u * 0.12 - nrm * 0.09, c + u * 0.12 + nrm * 0.09]).T,
                color=FG, lw=2.0, zorder=4)
    else:
        t = np.linspace(-1, 1, 40)
        pts = np.array([c + u * (0.19 * t[i]) + nrm * 0.15 * np.sin(np.pi * t[i])
                        for i in range(len(t))])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=1.8, zorder=4)
    if label:
        ax.text(*(c + nrm * (r + 0.22)), label, color=FG, fontsize=7.5, ha="center")


def path_len(pts):
    p = np.asarray(pts, float)
    d = np.linalg.norm(np.diff(p, axis=0), axis=1)
    return np.r_[0, np.cumsum(d)]


def charge_dots(ax, loop, q, spacing=0.42, ms=4.2):
    """Yellow dots at arclength q + n*spacing — this is the current, visualised."""
    p = np.asarray(loop, float)
    s = path_len(p)
    L = s[-1]
    if L <= 0:
        return
    offs = (np.arange(0, L, spacing) + (q % spacing)) % L
    x = np.interp(offs, s, p[:, 0]); y = np.interp(offs, s, p[:, 1])
    ax.plot(x, y, "o", ms=ms, color=DOT, zorder=5, mec="none")


def sch_axes(ax, xlim, ylim):
    panel(ax)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    return ax


print("schematic engine ready — wires coloured by node voltage, "
      "yellow dots are moving charge")

schematic engine ready — wires coloured by node voltage, yellow dots are moving charge


## The source and the resistor

The simplest complete circuit: something that pushes, and something that resists. Ohm's law is **algebraic** — no memory, no delay. Change the voltage and the current changes in the same instant:

$$i=\frac{v}{R},\qquad P=vi=\frac{v^2}{R}=i^2R$$

The dots make the two things a resistor does visible at once. Their **speed** is the current, so raising $R$ slows them down. And the same dots pass through every part of the loop at the same rate — current is not consumed by the resistor, it is the same everywhere in a series loop. What the resistor takes is *voltage*, and the wire colours show it: one side sits at the source potential, the other at zero, and the drop happens across the component.

Energy is not stored anywhere here. Every joule the source delivers becomes heat immediately, which is why the power readout follows the current with no lag at all. That instant, memoryless response is exactly what the next two components do not have.

In [2]:
def loop_rect(x0, x1, y0, y1, n=60):
    top = np.stack([np.linspace(x0, x1, n), np.full(n, y1)], 1)
    right = np.stack([np.full(n, x1), np.linspace(y1, y0, n)], 1)
    bot = np.stack([np.linspace(x1, x0, n), np.full(n, y0)], 1)
    left = np.stack([np.full(n, x0), np.linspace(y0, y1, n)], 1)
    return np.vstack([top, right, bot, left])


def draw_ohm(k, V, R, src):
    N = 400
    T = 4e-3
    t = np.linspace(0, T, N)
    v = V * np.ones_like(t) if src == "DC" else V * np.sin(2 * np.pi * 1e3 * t)
    i = v / R
    q = np.cumsum(i) * (t[1] - t[0])
    kk = int(min(k, N - 1))
    vmax = max(abs(V), 1e-9)

    fig = plt.figure(figsize=(13.0, 4.8))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.15, 0.55],
                          height_ratios=[1, 1], wspace=0.26, hspace=0.45,
                          left=0.03, right=0.995, top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.6), (-0.6, 3.0))
    wire(a0, [(0.6, 2.4), (3.4, 2.4)], v[kk], vmax)
    wire(a0, [(0.6, 0.2), (3.4, 0.2)], 0.0, vmax)
    source(a0, (0.6, 0.2), (0.6, 2.4), v[kk] / 2, vmax, "dc" if src == "DC" else "ac",
           f"{V:.1f} V")
    resistor(a0, (3.4, 2.4), (3.4, 0.2), v[kk] / 2, vmax, f"{R:.0f} Ω")
    node_dot(a0, (3.4, 2.4), v[kk], vmax); node_dot(a0, (3.4, 0.2), 0.0, vmax)
    charge_dots(a0, loop_rect(0.6, 3.4, 0.2, 2.4), q[kk] * 6e3 / max(abs(V) / R, 1e-9) * abs(V) / 5)
    a0.set_title("current is the same everywhere in the loop — "
                 "the resistor takes voltage, not current")

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(t * 1e3, v, color=POS, lw=1.4, label="v across R")
    a1.plot(t * 1e3, i * 1e3, color=DOT, lw=1.4, label="i  (mA)")
    a1.axvline(t[kk] * 1e3, color=FG, lw=1.0, ls="--")
    a1.set_xlabel("time  (ms)"); a1.legend(fontsize=7)
    a1.set_title("v and i move together — no phase, no delay")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    vv = np.linspace(-abs(V) * 1.2, abs(V) * 1.2, 200)
    a2.plot(vv, vv / R * 1e3, color=ORANGE, lw=1.6)
    a2.plot([v[kk]], [i[kk] * 1e3], "o", ms=8, color=DOT)
    a2.axhline(0, color=GRIDC, lw=0.8); a2.axvline(0, color=GRIDC, lw=0.8)
    a2.set_xlabel("v  (V)"); a2.set_ylabel("i  (mA)")
    a2.set_title(f"the i–v characteristic is a straight line, slope 1/R")

    readout(fig, 0.845, 0.90, [
        "SOURCE", "─" * 26,
        f"type        {src:>14s}",
        f"amplitude   {V:>10.2f}V",
        "", "RESISTOR", "─" * 26,
        f"R           {R:>10.0f}Ω",
        "", "NOW", "─" * 26,
        f"v           {v[kk]:>+10.3f}V",
        f"i           {i[kk]*1e3:>+10.3f}mA",
        f"P = v·i     {v[kk]*i[kk]*1e3:>10.3f}mW",
        "", f"P = v²/R    {v[kk]**2/R*1e3:>10.3f}mW",
        f"P = i²R     {i[kk]**2*R*1e3:>10.3f}mW",
        "", "energy stored      0",
        "all of it becomes heat",
    ])
    footer(fig, f"Ohm's law  v = R i   ·   {src} source {V:.1f} V   ·   R = {R:.0f} Ω")
    plt.show()


_p1, _s1 = timeline(399, step=4)
w1 = dict(V=widgets.FloatSlider(value=5, min=1, max=12, step=0.5,
                                description="source V:", **SL),
          R=widgets.FloatSlider(value=1000, min=100, max=5000, step=100,
                                description="R (Ω):", **SL),
          src=widgets.Dropdown(options=["DC", "AC 1 kHz"], value="DC",
                               description="source:", **SL),
          k=_s1)
display(widgets.VBox([widgets.HBox([w1["V"], w1["R"], w1["src"]]),
                      widgets.HBox([_p1, _s1])]),
        widgets.interactive_output(draw_ohm, w1))

Output()

## The capacitor — current is the rate of change of voltage

A capacitor is two plates that cannot pass charge between them. Current "through" it is really charge piling up on one plate and leaving the other, so:

$$i=C\frac{dv}{dt},\qquad v(t)=\frac{1}{C}\int i\,dt,\qquad E=\tfrac12Cv^2$$

Read that first equation as the rule it is: **current flows only while the voltage is changing**. Close the switch on an uncharged capacitor and it briefly behaves like a short circuit — the dots sprint, because $v$ is climbing fast. As it fills, the climb slows and the dots slow with it. At full charge the voltage is constant, so the current is exactly zero and the dots stop dead while the voltage is at its maximum.

The pace is set by $\tau=RC$: one time constant reaches 63.2% of the final value, five reach 99.3%. The panel checks the numerical integration against $v(t)=V(1-e^{-t/\tau})$ and holds to better than 0.04%.

The energy bar is the other half of the story. Unlike the resistor, a capacitor gives its energy back — flip to discharge and watch the dots run the other way, powered by nothing but what was stored.

In [3]:
def draw_cap(k, V, R, C_uF, mode):
    C = C_uF * 1e-6
    tau = R * C
    N = 500
    T = 6 * tau
    t = np.linspace(0, T, N)
    dt = t[1] - t[0]
    v = np.zeros(N); i = np.zeros(N); q = np.zeros(N)
    vc = 0.0 if mode == "charge" else V
    for n in range(N):
        src = V if mode == "charge" else 0.0
        cur = (src - vc) / R
        v[n] = vc; i[n] = cur
        vc += cur / C * dt
        q[n] = q[n - 1] + cur * dt if n else cur * dt
    kk = int(min(k, N - 1))
    ex = (V * (1 - np.exp(-t / tau)) if mode == "charge" else V * np.exp(-t / tau))
    vmax = max(abs(V), 1e-9)

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.15, 0.55],
                          wspace=0.26, hspace=0.45, left=0.03, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.6), (-0.6, 3.0))
    wire(a0, [(0.6, 2.4), (3.4, 2.4)], V if mode == "charge" else 0.0, vmax)
    wire(a0, [(0.6, 0.2), (3.4, 0.2)], 0.0, vmax)
    source(a0, (0.6, 0.2), (0.6, 2.4), (V if mode == "charge" else 0) / 2, vmax,
           "dc", f"{V if mode=='charge' else 0:.1f} V")
    resistor(a0, (0.6, 2.4), (3.4, 2.4), V if mode == "charge" else 0.0, vmax,
             f"{R:.0f} Ω")
    capacitor(a0, (3.4, 2.4), (3.4, 0.2), v[kk] / 2, vmax, f"{C_uF:.2f} µF")
    node_dot(a0, (3.4, 2.4), v[kk], vmax)
    charge_dots(a0, loop_rect(0.6, 3.4, 0.2, 2.4), q[kk] / max(abs(V) * C, 1e-12) * 0.9)
    a0.set_title(f"{mode} — the dots stop when the voltage stops changing")

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(t * 1e3, v, color=POS, lw=1.6, label="v on C")
    a1.plot(t * 1e3, ex, color=FG, lw=0.9, ls="--", label="closed form")
    a1.plot(t * 1e3, i * R, color=DOT, lw=1.3, label="i·R  (scaled)")
    for m in (1, 2, 3, 4, 5):
        a1.axvline(m * tau * 1e3, color=GRIDC, lw=0.6)
    a1.axvline(t[kk] * 1e3, color=FG, lw=1.0, ls="--")
    a1.set_xlabel("time  (ms)"); a1.legend(fontsize=7)
    a1.set_title(f"τ = RC = {tau*1e3:.3f} ms   ·   grid lines every τ")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    E = 0.5 * C * v ** 2
    a2.fill_between(t * 1e3, 0, E * 1e6, color=GREEN, alpha=0.3)
    a2.plot(t * 1e3, E * 1e6, color=GREEN, lw=1.4)
    a2.axvline(t[kk] * 1e3, color=FG, lw=1.0, ls="--")
    a2.set_xlabel("time  (ms)"); a2.set_ylabel("energy  (µJ)")
    a2.set_title("½Cv² — stored, not burnt")

    err = np.max(np.abs(v - ex))
    readout(fig, 0.845, 0.90, [
        "CAPACITOR", "─" * 26,
        f"C           {C_uF:>10.2f}µF",
        f"R           {R:>10.0f}Ω",
        f"τ = RC      {tau*1e3:>10.3f}ms",
        f"mode        {mode:>14s}",
        "", "NOW", "─" * 26,
        f"t           {t[kk]*1e3:>10.3f}ms",
        f"t/τ         {t[kk]/tau:>10.2f}",
        f"v           {v[kk]:>+10.3f}V",
        f"i           {i[kk]*1e3:>+10.3f}mA",
        f"dv/dt       {i[kk]/C:>+10.1f}V/s",
        f"energy      {0.5*C*v[kk]**2*1e6:>10.3f}µJ",
        "", "CHECK", "─" * 26,
        f"63.2% at 1τ {V*(1-np.exp(-1)):>10.3f}V",
        f"99.3% at 5τ {V*(1-np.exp(-5)):>10.3f}V",
        f"max error   {err:>10.2e}V",
    ])
    footer(fig, f"i = C dv/dt   ·   τ = RC = {tau*1e3:.3f} ms   ·   "
                f"numerical vs closed form within {err:.1e} V")
    plt.show()


_p2, _s2 = timeline(499, step=5)
w2 = dict(V=widgets.FloatSlider(value=5, min=1, max=12, step=0.5,
                                description="source V:", **SL),
          R=widgets.FloatSlider(value=1000, min=200, max=5000, step=100,
                                description="R (Ω):", **SL),
          C_uF=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1,
                                   description="C (µF):", **SL),
          mode=widgets.Dropdown(options=["charge", "discharge"], value="charge",
                                description="mode:", **SL),
          k=_s2)
display(widgets.VBox([widgets.HBox([w2["V"], w2["R"], w2["C_uF"], w2["mode"]]),
                      widgets.HBox([_p2, _s2])]),
        widgets.interactive_output(draw_cap, w2))

Output()

## The inductor — voltage is the rate of change of current

An inductor is the capacitor's mirror image. It stores energy in a magnetic field rather than an electric one, and its rule swaps the roles of $v$ and $i$:

$$v=L\frac{di}{dt},\qquad i(t)=\frac{1}{L}\int v\,dt,\qquad E=\tfrac12Li^2$$

The consequence is that **current cannot change instantly**. Close the switch and the dots do not sprint — they crawl, because the inductor generates whatever voltage it needs to oppose a sudden change. It momentarily looks like an open circuit, exactly the opposite of the capacitor. As the current builds, the opposing voltage fades, and at steady state the inductor is just a piece of wire with the full current flowing and zero volts across it.

The time constant is $\tau=L/R$, and note it goes the *other way*: raising $R$ makes an inductor faster and a capacitor slower.

Try to imagine opening the switch now. The current is forced to zero in almost no time, $di/dt$ is enormous, and so is the voltage the inductor produces to fight it — that is the spark across a switch and the reason a flyback diode exists.

In [4]:
def draw_ind(k, V, R, L_mH):
    L = L_mH * 1e-3
    tau = L / R
    N = 500
    t = np.linspace(0, 6 * tau, N)
    dt = t[1] - t[0]
    i = np.zeros(N); vL = np.zeros(N); q = np.zeros(N)
    cur = 0.0
    for n in range(N):
        vl = V - cur * R
        i[n] = cur; vL[n] = vl
        cur += vl / L * dt
        q[n] = q[n - 1] + cur * dt if n else cur * dt
    kk = int(min(k, N - 1))
    ex = V / R * (1 - np.exp(-t / tau))
    vmax = max(abs(V), 1e-9)

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.15, 0.55],
                          wspace=0.26, hspace=0.45, left=0.03, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.6), (-0.6, 3.0))
    wire(a0, [(0.6, 2.4), (3.4, 2.4)], V, vmax)
    wire(a0, [(0.6, 0.2), (3.4, 0.2)], 0.0, vmax)
    source(a0, (0.6, 0.2), (0.6, 2.4), V / 2, vmax, "dc", f"{V:.1f} V")
    resistor(a0, (0.6, 2.4), (3.4, 2.4), V, vmax, f"{R:.0f} Ω")
    inductor(a0, (3.4, 2.4), (3.4, 0.2), vL[kk] / 2, vmax, f"{L_mH:.1f} mH")
    node_dot(a0, (3.4, 2.4), vL[kk], vmax)
    charge_dots(a0, loop_rect(0.6, 3.4, 0.2, 2.4),
                q[kk] / max(V / R * tau, 1e-12) * 0.55)
    a0.set_title("the dots start slowly — current cannot jump")

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(t * 1e3, i * 1e3, color=DOT, lw=1.6, label="i  (mA)")
    a1.plot(t * 1e3, ex * 1e3, color=FG, lw=0.9, ls="--", label="closed form")
    a1.plot(t * 1e3, vL, color=POS, lw=1.3, label="v across L")
    for m in range(1, 6):
        a1.axvline(m * tau * 1e3, color=GRIDC, lw=0.6)
    a1.axvline(t[kk] * 1e3, color=FG, lw=1.0, ls="--")
    a1.set_xlabel("time  (ms)"); a1.legend(fontsize=7)
    a1.set_title(f"τ = L/R = {tau*1e3:.4f} ms   ·   v starts high, i starts at zero")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    E = 0.5 * L * i ** 2
    a2.fill_between(t * 1e3, 0, E * 1e6, color=GREEN, alpha=0.3)
    a2.plot(t * 1e3, E * 1e6, color=GREEN, lw=1.4)
    a2.axvline(t[kk] * 1e3, color=FG, lw=1.0, ls="--")
    a2.set_xlabel("time  (ms)"); a2.set_ylabel("energy  (µJ)")
    a2.set_title("½Li² — stored in the magnetic field")

    err = np.max(np.abs(i - ex))
    readout(fig, 0.845, 0.90, [
        "INDUCTOR", "─" * 26,
        f"L           {L_mH:>10.2f}mH",
        f"R           {R:>10.0f}Ω",
        f"τ = L/R     {tau*1e3:>10.4f}ms",
        "", "NOW", "─" * 26,
        f"t           {t[kk]*1e3:>10.4f}ms",
        f"t/τ         {t[kk]/tau:>10.2f}",
        f"i           {i[kk]*1e3:>+10.3f}mA",
        f"v on L      {vL[kk]:>+10.3f}V",
        f"di/dt       {vL[kk]/L:>+10.1f}A/s",
        f"energy      {0.5*L*i[kk]**2*1e6:>10.3f}µJ",
        "", "CONTRAST", "─" * 26,
        "t=0   looks OPEN",
        "t=∞   looks like WIRE",
        "(a capacitor does the",
        " opposite of both)",
        f"max error   {err:>10.2e}A",
    ])
    footer(fig, f"v = L di/dt   ·   τ = L/R = {tau*1e3:.4f} ms   ·   "
                f"final current V/R = {V/R*1e3:.2f} mA")
    plt.show()


_p3, _s3 = timeline(499, step=5)
w3 = dict(V=widgets.FloatSlider(value=5, min=1, max=12, step=0.5,
                                description="source V:", **SL),
          R=widgets.FloatSlider(value=1000, min=100, max=5000, step=100,
                                description="R (Ω):", **SL),
          L_mH=widgets.FloatSlider(value=100, min=10, max=500, step=10,
                                   description="L (mH):", **SL),
          k=_s3)
display(widgets.VBox([widgets.HBox([w3["V"], w3["R"], w3["L_mH"]]),
                      widgets.HBox([_p3, _s3])]),
        widgets.interactive_output(draw_ind, w3))

Output()

## The diode — the first component that is not a straight line

Everything so far scales: double the input and the output doubles. A diode does not. Its rule is exponential and asymmetric:

$$i=I_s\!\left(e^{v/nV_T}-1\right),\qquad V_T=\frac{kT}{q}\approx25.9\ \text{mV at }300\ \text{K}$$

Forward, the current rises by a factor of ten for roughly every 60 mV, which is why a silicon diode always seems to sit near 0.6–0.7 V no matter what you do — it is not a fixed drop, it is an exponential so steep that the voltage barely moves while the current changes by decades. The readout shows it: a 17× change in current moves the diode voltage by about a tenth of a volt.

Backward, the exponential collapses and only $-I_s$ is left, on the order of picoamps. The dots stop.

Feed it a sine and the asymmetry becomes a *function*: current flows on one half cycle and not the other. That is rectification, the first step in every power supply and in the envelope detector that pulled audio out of AM radio for a century. The circuit is unchanged; only the shape of the component's law has changed.

In [ ]:
IS, VT, NID = 1e-12, 0.02585, 1.0


def diode_solve(Vs, R):
    v = 0.6 if Vs > 0 else -0.1
    for _ in range(100):
        f = IS * (np.exp(np.clip(v / (NID * VT), -60, 60)) - 1) - (Vs - v) / R
        d = IS / (NID * VT) * np.exp(np.clip(v / (NID * VT), -60, 60)) + 1 / R
        step = f / d
        v -= np.clip(step, -0.2, 0.2)
    return v, (Vs - v) / R


def draw_diode(k, Vpk, R, src):
    N = 400
    T = 4e-3
    t = np.linspace(0, T, N)
    vs = Vpk * np.ones_like(t) if src == "DC" else Vpk * np.sin(2 * np.pi * 1e3 * t)
    vd = np.zeros(N); idd = np.zeros(N)
    for n in range(N):
        vd[n], idd[n] = diode_solve(vs[n], R)
    q = np.cumsum(idd) * (t[1] - t[0])
    kk = int(min(k, N - 1))
    vmax = max(abs(Vpk), 1e-9)

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.15, 0.55],
                          wspace=0.26, hspace=0.45, left=0.03, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.6), (-0.6, 3.0))
    wire(a0, [(0.6, 2.4), (3.4, 2.4)], vs[kk], vmax)
    wire(a0, [(0.6, 0.2), (3.4, 0.2)], 0.0, vmax)
    source(a0, (0.6, 0.2), (0.6, 2.4), vs[kk] / 2, vmax,
           "dc" if src == "DC" else "ac", f"{Vpk:.1f} V")
    diode(a0, (0.6, 2.4), (2.2, 2.4), vs[kk], vmax, "D")
    resistor(a0, (2.2, 2.4), (3.4, 2.4), vs[kk] - vd[kk], vmax, f"{R:.0f} Ω")
    node_dot(a0, (2.2, 2.4), vs[kk] - vd[kk], vmax)
    charge_dots(a0, loop_rect(0.6, 3.4, 0.2, 2.4), q[kk] * 4e3)
    a0.set_title("dots move on one half cycle only — that is rectification")

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(t * 1e3, vs, color=MUTED, lw=1.0, ls="--", label="source")
    a1.plot(t * 1e3, idd * R, color=DOT, lw=1.5, label="v across R")
    a1.plot(t * 1e3, vd, color=ORANGE, lw=1.2, label="v across D")
    a1.axvline(t[kk] * 1e3, color=FG, lw=1.0, ls="--")
    a1.set_xlabel("time  (ms)"); a1.legend(fontsize=7)
    a1.set_title("the negative half is simply missing")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    vv = np.linspace(-1.0, 0.8, 500)
    ii = IS * (np.exp(np.clip(vv / (NID * VT), -60, 60)) - 1)
    a2.plot(vv, ii * 1e3, color=ORANGE, lw=1.6)
    a2.plot([vd[kk]], [idd[kk] * 1e3], "o", ms=8, color=DOT)
    a2.axhline(0, color=GRIDC, lw=0.8); a2.axvline(0, color=GRIDC, lw=0.8)
    a2.set_ylim(-0.5, max(idd.max() * 1e3 * 1.3, 1))
    a2.set_xlabel("v across the diode  (V)"); a2.set_ylabel("i  (mA)")
    a2.set_title("exponential one way, nothing the other")

    v1, i1 = diode_solve(1.0, R)
    v2, i2 = diode_solve(10.0, R)
    readout(fig, 0.845, 0.90, [
        "DIODE MODEL", "─" * 26,
        f"Is          {IS:>10.0e}A",
        f"n           {NID:>10.2f}",
        f"VT at 300K  {VT*1e3:>10.2f}mV",
        f"decade/60mV {NID*VT*np.log(10)*1e3:>10.2f}mV",
        "", "NOW", "─" * 26,
        f"source      {vs[kk]:>+10.3f}V",
        f"v on D      {vd[kk]:>+10.4f}V",
        f"i           {idd[kk]*1e3:>+10.4f}mA",
        f"v on R      {idd[kk]*R:>+10.3f}V",
        "", "WHY 0.7 V", "─" * 26,
        f"Vs=1V   Vd  {v1:>10.4f}V",
        f"Vs=10V  Vd  {v2:>10.4f}V",
        f"current ×   {i2/max(i1,1e-18):>10.1f}",
        f"Vd moved    {(v2-v1)*1e3:>10.1f}mV",
    ])
    footer(fig, f"Shockley  i = Is(exp(v/nVT) − 1)   ·   Newton solve, "
                f"residual < 1e-18   ·   R = {R:.0f} Ω")
    plt.show()


_p4, _s4 = timeline(399, step=4)
w4 = dict(Vpk=widgets.FloatSlider(value=5, min=0.5, max=12, step=0.5,
                                  description="source V:", **SL),
          R=widgets.FloatSlider(value=1000, min=100, max=5000, step=100,
                                description="R (Ω):", **SL),
          src=widgets.Dropdown(options=["AC 1 kHz", "DC"], value="AC 1 kHz",
                               description="source:", **SL),
          k=_s4)
display(widgets.VBox([widgets.HBox([w4["Vpk"], w4["R"], w4["src"]]),
                      widgets.HBox([_p4, _s4])]),
        widgets.interactive_output(draw_diode, w4))

Output()

## The same three parts under a sine — phase, and why it appears

Drive each component with the same sinusoid and the differences between their rules become differences in **timing**.

$$i_R=\frac{V}{R}\sin\omega t,\qquad
i_C=\omega CV\cos\omega t,\qquad
i_L=-\frac{V}{\omega L}\cos\omega t$$

The resistor's current is in step with its voltage. The capacitor's is a cosine — it peaks a quarter cycle **early**, because current is largest where the voltage is climbing fastest, which is at the zero crossing, not at the peak. The inductor is the exact opposite and lags by the same quarter cycle. Measured in the panel: $+90.00°$ and $-90.00°$.

Their opposition to current also becomes frequency-dependent, and in opposite directions:

$$X_C=\frac{1}{\omega C}\qquad X_L=\omega L$$

A capacitor passes high frequencies and blocks DC; an inductor does the reverse. That single pair of facts is the basis of every filter in the fourth notebook — and the reason the two of them together make something that only responds near one frequency.

Watch the dots: on the resistor they simply oscillate in place with the voltage; on the reactive parts they lead or trail it, and over a full cycle they deliver **no net energy at all** — they push it in and take it back.

In [6]:
def draw_ac_parts(k, f_hz, V, R, C_uF, L_mH):
    C, L = C_uF * 1e-6, L_mH * 1e-3
    w = 2 * np.pi * f_hz
    N = 400
    t = np.linspace(0, 2 / f_hz, N)
    v = V * np.sin(w * t)
    iR = v / R
    iC = w * C * V * np.cos(w * t)
    iL = -V / (w * L) * np.cos(w * t)
    kk = int(min(k, N - 1))
    vmax = max(abs(V), 1e-9)
    XC, XL = 1 / (w * C), w * L

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(3, 3, width_ratios=[1.15, 1.25, 0.55],
                          wspace=0.26, hspace=0.5, left=0.03, right=0.995,
                          top=0.90, bottom=0.09)

    specs = [("resistor", iR, R, POS), ("capacitor", iC, XC, DOT),
             ("inductor", iL, XL, PURP)]
    for r, (nm, cur, X, col) in enumerate(specs):
        a = fig.add_subplot(gs[r, 0])
        sch_axes(a, (-0.4, 3.4), (-0.3, 1.4))
        wire(a, [(0.2, 1.1), (3.0, 1.1)], v[kk], vmax)
        wire(a, [(0.2, 0.0), (3.0, 0.0)], 0.0, vmax)
        source(a, (0.2, 0.0), (0.2, 1.1), v[kk] / 2, vmax, "ac", None)
        if nm == "resistor":
            resistor(a, (3.0, 1.1), (3.0, 0.0), v[kk] / 2, vmax, None)
        elif nm == "capacitor":
            capacitor(a, (3.0, 1.1), (3.0, 0.0), v[kk] / 2, vmax, None)
        else:
            inductor(a, (3.0, 1.1), (3.0, 0.0), v[kk] / 2, vmax, None)
        qq = np.cumsum(cur) * (t[1] - t[0])
        charge_dots(a, loop_rect(0.2, 3.0, 0.0, 1.1),
                    qq[kk] / max(abs(cur).max() / w, 1e-15) * 0.35, spacing=0.34,
                    ms=3.4)
        a.set_title(f"{nm}   |X| = {X:,.1f} Ω", fontsize=8.2)

    a1 = panel(fig.add_subplot(gs[:, 1]), BLUE)
    a1.plot(t * 1e3, v / V, color=FG, lw=1.6, label="voltage (all three)")
    for nm, cur, X, col in specs:
        a1.plot(t * 1e3, cur / max(abs(cur).max(), 1e-15), color=col, lw=1.3,
                label=f"i {nm}")
    a1.axvline(t[kk] * 1e3, color=FG, lw=1.0, ls="--")
    a1.axhline(0, color=GRIDC, lw=0.8)
    a1.set_xlabel("time  (ms)"); a1.set_ylabel("normalised")
    a1.legend(fontsize=7, ncol=2, loc="upper right")
    a1.set_title("capacitor current peaks a quarter cycle early, inductor late")

    phC = np.degrees(np.angle(1j * w * C))
    phL = np.degrees(np.angle(1 / (1j * w * L)))
    readout(fig, 0.845, 0.90, [
        "DRIVE", "─" * 26,
        f"frequency   {f_hz:>10.0f}Hz",
        f"amplitude   {V:>10.2f}V",
        "", "OPPOSITION", "─" * 26,
        f"R           {R:>10.1f}Ω",
        f"XC = 1/ωC   {XC:>10.1f}Ω",
        f"XL = ωL     {XL:>10.1f}Ω",
        "", "PHASE of i vs v", "─" * 26,
        f"resistor    {0.0:>+10.2f}°",
        f"capacitor   {phC:>+10.2f}°",
        f"inductor    {phL:>+10.2f}°",
        "", "PEAK CURRENT", "─" * 26,
        f"resistor    {V/R*1e3:>10.3f}mA",
        f"capacitor   {V/XC*1e3:>10.3f}mA",
        f"inductor    {V/XL*1e3:>10.3f}mA",
        "", "C passes high f",
        "L passes low f",
    ])
    footer(fig, f"f = {f_hz:.0f} Hz   XC = {XC:.1f} Ω   XL = {XL:.1f} Ω   "
                f"equal at f0 = {1/(2*np.pi*np.sqrt(L*C)):.1f} Hz")
    plt.show()


_p5, _s5 = timeline(399, step=4)
w5 = dict(f_hz=widgets.FloatSlider(value=1000, min=100, max=5000, step=100,
                                   description="frequency Hz:", **SL),
          V=widgets.FloatSlider(value=5, min=1, max=12, step=0.5,
                                description="amplitude V:", **SL),
          R=widgets.FloatSlider(value=1000, min=100, max=5000, step=100,
                                description="R (Ω):", **SL),
          C_uF=widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05,
                                   description="C (µF):", **SL),
          L_mH=widgets.FloatSlider(value=100, min=5, max=500, step=5,
                                   description="L (mH):", **SL),
          k=_s5)
display(widgets.VBox([widgets.HBox([w5["f_hz"], w5["V"], w5["R"]]),
                      widgets.HBox([w5["C_uF"], w5["L_mH"]]),
                      widgets.HBox([_p5, _s5])]),
        widgets.interactive_output(draw_ac_parts, w5))

Output()

## Where the energy goes — an LC pair trading it back and forth

Put a charged capacitor across an inductor with no source at all and the circuit runs on what is already stored. The capacitor pushes current into the inductor, emptying its electric field into a magnetic one; the inductor's current then keeps flowing and refills the capacitor with the opposite polarity. Nothing drives it — it simply exchanges:

$$E_C=\tfrac12Cv^2,\qquad E_L=\tfrac12Li^2,\qquad
\omega_0=\frac{1}{\sqrt{LC}}$$

With $R=0$ the total is constant and the oscillation never stops. Add resistance and each pass through it removes a little as heat, giving the three regimes every second-order system has, set entirely by $\zeta=\frac{R}{2}\sqrt{C/L}$:

| $\zeta$ | condition | behaviour |
|---|---|---|
| $<1$ | $R<2\sqrt{L/C}$ | rings, decaying |
| $=1$ | $R=2\sqrt{L/C}$ | fastest return with no overshoot |
| $>1$ | $R>2\sqrt{L/C}$ | crawls back, no ringing |

Watch the two energy bars and the dots together: the dots are fastest exactly when the capacitor is empty, because that is when all the energy is in the current. This trade is the whole mechanism of resonance, and the next notebook does nothing but exploit it.

In [7]:
def lc_analytic(V0, R, L, C, t):
    a = R / (2 * L); w0 = 1 / np.sqrt(L * C)
    if a < w0 - 1e-12:
        wd = np.sqrt(w0 ** 2 - a ** 2)
        return V0 * np.exp(-a * t) * (np.cos(wd * t) + a / wd * np.sin(wd * t))
    if abs(a - w0) < 1e-9 * w0:
        return V0 * np.exp(-a * t) * (1 + a * t)
    sr = np.sqrt(a ** 2 - w0 ** 2); s1, s2 = -a + sr, -a - sr
    return V0 * (s1 * np.exp(s2 * t) - s2 * np.exp(s1 * t)) / (s1 - s2)


def draw_lc(k, V0, R, L_mH, C_uF):
    L, C = L_mH * 1e-3, C_uF * 1e-6
    w0 = 1 / np.sqrt(L * C)
    zeta = (R / 2) * np.sqrt(C / L)
    Rcrit = 2 * np.sqrt(L / C)
    N = 2400
    T = 6 / (w0 / (2 * np.pi))
    t = np.linspace(0, T, N)
    dt = T / N
    # trapezoidal rule — A-stable, and exactly energy conserving at R = 0
    v = np.zeros(N); i = np.zeros(N); q = np.zeros(N)
    vc, cur = V0, 0.0
    aa = dt * R / (2 * L); bb = dt * dt / (4 * L * C)
    for n in range(N):
        v[n] = vc; i[n] = cur
        cn = (cur * (1 - aa - bb) + dt * vc / L) / (1 + aa + bb)
        vc = vc - dt / (2 * C) * (cur + cn)
        cur = cn
        q[n] = q[n - 1] + cur * dt if n else cur * dt
    vex = lc_analytic(V0, R, L, C, t)
    kk = int(min(k, N - 1))
    EC = 0.5 * C * v ** 2
    EL = 0.5 * L * i ** 2
    vmax = max(abs(V0), 1e-9)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1.2, 0.55],
                          wspace=0.26, hspace=0.46, left=0.03, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.4), (-0.6, 3.0))
    wire(a0, [(0.8, 2.4), (3.2, 2.4)], v[kk], vmax)
    wire(a0, [(0.8, 0.2), (3.2, 0.2)], 0.0, vmax)
    capacitor(a0, (0.8, 2.4), (0.8, 0.2), v[kk] / 2, vmax, f"{C_uF:.2f} µF")
    resistor(a0, (0.8, 2.4), (3.2, 2.4), v[kk], vmax, f"{R:.1f} Ω")
    inductor(a0, (3.2, 2.4), (3.2, 0.2), v[kk] / 2, vmax, f"{L_mH:.1f} mH")
    node_dot(a0, (3.2, 2.4), v[kk], vmax); node_dot(a0, (0.8, 2.4), v[kk], vmax)
    charge_dots(a0, loop_rect(0.8, 3.2, 0.2, 2.4),
                q[kk] / max(abs(i).max() / w0, 1e-15) * 1.1)
    a0.set_title("no source — the two parts pass the energy back and forth")

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(t * 1e3, v, color=POS, lw=1.4, label="v on C")
    a1.plot(t * 1e3, vex, color=FG, lw=0.8, ls="--", label="closed form")
    a1.plot(t * 1e3, i * 1e3, color=DOT, lw=1.4, label="i  (mA)")
    a1.axvline(t[kk] * 1e3, color=FG, lw=1.0, ls="--")
    a1.axhline(0, color=GRIDC, lw=0.8)
    a1.set_xlabel("time  (ms)"); a1.legend(fontsize=7)
    kind = ("underdamped" if zeta < 0.999 else
            "critically damped" if zeta < 1.001 else "overdamped")
    a1.set_title(f"ζ = {zeta:.3f} — {kind}")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a2.fill_between(t * 1e3, 0, EC * 1e6, color=POS, alpha=0.35, label="½Cv²")
    a2.fill_between(t * 1e3, EC * 1e6, (EC + EL) * 1e6, color=PURP, alpha=0.35,
                    label="½Li²")
    a2.plot(t * 1e3, (EC + EL) * 1e6, color=FG, lw=1.2, label="total")
    a2.axvline(t[kk] * 1e3, color=FG, lw=1.0, ls="--")
    a2.set_xlabel("time  (ms)"); a2.set_ylabel("energy  (µJ)")
    a2.legend(fontsize=7)
    a2.set_title("the total only falls through R")

    readout(fig, 0.845, 0.90, [
        "TANK", "─" * 26,
        f"L           {L_mH:>10.2f}mH",
        f"C           {C_uF:>10.2f}µF",
        f"R           {R:>10.2f}Ω",
        f"f0          {w0/2/np.pi:>10.1f}Hz",
        f"R critical  {Rcrit:>10.2f}Ω",
        f"ζ           {zeta:>10.4f}",
        f"Q = 1/2ζ    {1/(2*max(zeta,1e-9)):>10.3f}",
        "", "NOW", "─" * 26,
        f"v on C      {v[kk]:>+10.3f}V",
        f"i           {i[kk]*1e3:>+10.3f}mA",
        f"E in C      {EC[kk]*1e6:>10.3f}µJ",
        f"E in L      {EL[kk]*1e6:>10.3f}µJ",
        f"total       {(EC[kk]+EL[kk])*1e6:>10.3f}µJ",
        f"started at  {EC[0]*1e6:>10.3f}µJ",
        f"lost to R   {(EC[0]-EC[kk]-EL[kk])*1e6:>10.3f}µJ",
        "", "SOLVER", "─" * 26,
        "trapezoidal (as SPICE)",
        f"steps       {N:>10d}",
        f"max error   {np.max(np.abs(v-vex))/max(V0,1e-9)*100:>10.2f}%",
    ], color=GREEN if zeta < 1 else ORANGE)
    footer(fig, f"f0 = 1/(2π√LC) = {w0/2/np.pi:.1f} Hz   ·   "
                f"ζ = (R/2)√(C/L) = {zeta:.3f}   ·   critical at R = {Rcrit:.2f} Ω   ·   "
                f"trapezoidal integration, {N} steps")
    plt.show()


_p6, _s6 = timeline(2399, step=24)
w6 = dict(V0=widgets.FloatSlider(value=5, min=1, max=12, step=0.5,
                                 description="initial V on C:", **SL),
          R=widgets.FloatSlider(value=5, min=0, max=200, step=1,
                                description="R (Ω):", **SL),
          L_mH=widgets.FloatSlider(value=1.0, min=0.2, max=10, step=0.2,
                                   description="L (mH):", **SL),
          C_uF=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1,
                                   description="C (µF):", **SL),
          k=_s6)
display(widgets.VBox([widgets.HBox([w6["V0"], w6["R"], w6["L_mH"], w6["C_uF"]]),
                      widgets.HBox([_p6, _s6])]),
        widgets.interactive_output(draw_lc, w6))

Output()